# E-Commerce Order Risk Classifier

**Goal:** Predict whether an order is `Normal`, `Return Risk`, or `Fraud Risk`.

This notebook is designed to be **beginner-friendly** — every step is explained clearly.

---

## What this notebook covers

| Step | Description |
|------|-------------|
| 1 | Load & Explore the Dataset (EDA) |
| 2 | Data Preprocessing |
| 3 | Handle Class Imbalance (SMOTE) |
| 4 | Train a Random Forest Classifier |
| 5 | Evaluate Model Performance |
| 6 | Feature Importance |
| 7 | Conclusion |

> **Dataset:** Synthetic E-Commerce Order Risk Dataset  
> **Target:** `risk_label` — Normal / Return Risk / Fraud Risk


## Step 1: Import Libraries

We start by importing all the tools we need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('All libraries imported successfully!')

## Step 2: Load & Explore the Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/synthetic-ecommerce-order-risk-dataset/synthetic_ecommerce_order_risk_dataset.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

### Basic Information & Missing Values

In [ ]:
print('Data Types:')
print(df.dtypes)
print(f'\nMissing values total: {df.isnull().sum().sum()}')
print('\nStatistical Summary:')
df.describe()

## Step 3: Exploratory Data Analysis (EDA)

### Target Variable Distribution

Our target column is `risk_label`. Let's see how many orders fall in each category.

> **Note:** The dataset is **imbalanced** — most orders are Normal.

In [ ]:
counts = df['risk_label'].value_counts()
print(counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#e67e22', '#e74c3c']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Order Risk Label Counts', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Risk Label Proportions', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable: risk_label', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Numerical Features by Risk Label

In [ ]:
num_features = ['order_value_eur', 'review_score', 'discount_rate',
                'previous_orders', 'shipping_distance_km']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

label_colors = {'Normal':'#2ecc71', 'Return Risk':'#e67e22', 'Fraud Risk':'#e74c3c'}

for i, feat in enumerate(num_features):
    for label, color in label_colors.items():
        subset = df[df['risk_label'] == label][feat]
        axes[i].hist(subset, bins=30, alpha=0.6, label=label, color=color)
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=9)

num_cols = df.select_dtypes(include='number').drop(columns=['is_fraud','is_returned'])
sns.heatmap(num_cols.corr(), ax=axes[5], cmap='coolwarm', annot=True,
            fmt='.2f', linewidths=0.5, annot_kws={'size': 7})
axes[5].set_title('Correlation Heatmap', fontweight='bold')

plt.suptitle('Feature Distributions by Risk Label', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Categorical Features vs Risk Label

In [ ]:
cat_features = ['payment_method', 'product_category', 'device_type', 'traffic_source']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    ct = df.groupby([feat, 'risk_label']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i],
            color=['#e74c3c', '#2ecc71', '#e67e22'], edgecolor='black', width=0.7)
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=13)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(title='Risk Label', fontsize=9)

plt.suptitle('Categorical Features vs Risk Label', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing

Machine Learning models only understand **numbers**, not text.  
We need to:
1. **Drop** ID and date columns (not useful for prediction)
2. **Drop** `is_fraud` and `is_returned` — these would **leak** the target answer!
3. **Encode** categorical text columns into numbers using `LabelEncoder`

In [ ]:
DROP_COLS = ['order_id', 'order_date', 'is_fraud', 'is_returned']
CATEGORICAL_COLS = ['country', 'device_type', 'traffic_source',
                    'payment_method', 'product_category']

df_processed = df.drop(columns=DROP_COLS).copy()

# Encode categorical columns
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le

# Encode target
le_target = LabelEncoder()
df_processed['risk_label'] = le_target.fit_transform(df_processed['risk_label'])

print('Target class encoding:')
for i, cls in enumerate(le_target.classes_):
    print(f'  {i} -> {cls}')

print(f'\nProcessed shape: {df_processed.shape}')
df_processed.head()

## Step 5: Train-Test Split

We split data: **80% for training** and **20% for testing**.  
`stratify=y` ensures each risk class is proportionally represented in both splits.

In [ ]:
X = df_processed.drop(columns=['risk_label'])
y = df_processed['risk_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Test samples     : {X_test.shape[0]:,}')
print('\nClass distribution in training set:')
for i, cls in enumerate(le_target.classes_):
    print(f'  {cls}: {(y_train == i).sum()}')

## Step 6: Handle Class Imbalance with SMOTE

**The problem:** Fraud Risk has only 447 samples vs 10,482 Normal.  
A biased model will just predict 'Normal' for everything.

**SMOTE** (Synthetic Minority Over-sampling Technique) creates **artificial samples**  
for minority classes to balance the training data.

> Important: Apply SMOTE **only on training data**, never on test data!

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('Class counts BEFORE SMOTE (training):')
for i, cls in enumerate(le_target.classes_):
    print(f'  {cls}: {(y_train == i).sum()}')

print('\nClass counts AFTER SMOTE (training):')
for i, cls in enumerate(le_target.classes_):
    print(f'  {cls}: {(y_train_res == i).sum()}')

print('\nTraining data is now balanced!')

## Step 7: Train the Random Forest Classifier

**Random Forest** = an ensemble of many Decision Trees.  
Each tree votes on a prediction, and the majority vote wins.  
It's robust, handles mixed data types well, and rarely overfits.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,   # 200 decision trees
    random_state=42,    # for reproducibility
    n_jobs=-1           # use all available CPU cores
)

rf_model.fit(X_train_res, y_train_res)
print('Model trained successfully!')
print(f'  Trees: {rf_model.n_estimators}')
print(f'  Features: {rf_model.n_features_in_}')

## Step 8: Evaluate Model Performance

| Metric | Meaning |
|--------|---------|
| **Accuracy** | % of all predictions that are correct |
| **Precision** | Of predicted positives, how many are truly positive |
| **Recall** | Of all actual positives, how many did we catch |
| **F1-Score** | Balance between Precision and Recall |

In [ ]:
y_pred = rf_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print()
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

### Confusion Matrix

The diagonal shows **correct predictions**. Off-diagonal cells show misclassifications.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

pred_series = pd.Series(le_target.inverse_transform(y_pred)).value_counts()
actual_series = pd.Series(le_target.inverse_transform(y_test.values)).value_counts()
classes = le_target.classes_
x = np.arange(len(classes))
width = 0.35
axes[1].bar(x - width/2, [actual_series.get(c, 0) for c in classes],
            width, label='Actual', color='steelblue', edgecolor='black')
axes[1].bar(x + width/2, [pred_series.get(c, 0) for c in classes],
            width, label='Predicted', color='salmon', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes)
axes[1].set_title('Actual vs Predicted Counts', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].set_ylabel('Number of Orders')
plt.tight_layout()
plt.show()

## Step 9: Feature Importance

Random Forest can tell us **which features mattered most** in its decisions.  
Higher importance score = more influential in predicting the risk label.

In [ ]:
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = ['#e74c3c' if v > 0.1 else '#3498db' if v > 0.05 else '#95a5a6'
             for v in feat_imp.values]
bars = ax.barh(feat_imp.index, feat_imp.values, color=colors_fi, edgecolor='black')

for bar, val in zip(bars, feat_imp.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(feat_imp.mean(), color='orange', linestyle='--',
           label=f'Mean ({feat_imp.mean():.3f})')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 5 Features:')
for feat, val in feat_imp.sort_values(ascending=False).head(5).items():
    print(f'  {feat:30s}: {val:.4f}')

## Conclusion

### Results Summary

| Metric | Value |
|--------|-------|
| **Overall Accuracy** | ~91% |
| **Normal class** | Very high precision & recall |
| **Return Risk** | Good recall — model catches most cases |
| **Fraud Risk** | Hard to detect — extreme class imbalance |

---

### Key Takeaways

1. **Class imbalance is a real challenge** — Fraud Risk is only ~3.7% of orders.  
   SMOTE helped the model learn minority class patterns during training.

2. **`review_score` is the most important feature** — customers with low scores  
   tend to have higher return and fraud risk.

3. **Random Forest** works well for tabular classification tasks with mixed data types.

4. **Fraud detection is hard** without explicit fraud signals in the data.  
   In a real-world system, you would add behavioral patterns, device fingerprinting, etc.

---

### What You Can Try Next

- Try **XGBoost** or **LightGBM** for potentially better performance
- Apply **GridSearchCV** or **Optuna** for hyperparameter tuning
- Engineer new features (e.g., `order_value / avg_order_value` ratio)
- Try a **two-stage classifier**: Normal vs Risky first, then Return vs Fraud

---

> *If you found this notebook helpful, please give it an Upvote — it helps others discover it!*